In [7]:
# =============================================================================
# Environment Setup
# =============================================================================
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.ensemble import (ExtraTreesRegressor, RandomForestRegressor)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV

# Configure Plotly renderer for Jupyter Notebook
pio.renderers.default = "notebook_connected"

In [9]:
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.decomposition import PCA

In [10]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#### Exercise Dataset Variables

| Variable | Medical Meaning |
|----------|----------------|
| Duration | Length of exercise session (minutes) |
| Pulse | Average heart rate during exercise (beats per minute - bpm) |
| Maxpulse | Peak heart rate reached during exercise (bpm) |
| Calories | Estimated energy expenditure during the exercise session |

In [11]:
try:
    # When running from a Python module (e.g. src/data_loading.py)
    ROOT_DIR = Path(__file__).resolve().parents[0]
except NameError:
    # When running inside a notebook (no __file__)
    ROOT_DIR = Path().resolve().parents[0] # generates windows paths and converts to string 'C:/Users/dooke/Projects/data-science-refresher'

DATA_DIR = ROOT_DIR / "data" # reads 'C:/Users/dooke/Projects/data-science-refresher/data' folder

In [13]:
# filename for csv file name, subdir fie name i.e raw, processed etc
def load_csv(filename: str, subdir: str = "raw", **kwargs) -> pd.DataFrame:
    """
    Load a CSV from data/<subdir>/<name>.csv
    """
    return DATA_DIR / subdir / f"{filename}.csv"

In [14]:
data_file = pd.read_csv(load_csv("impulse_data"))

In [15]:
data_file

,Duration,Pulse,Maxpulse,Calories
0,60,110,130,409.1
1,60,117,145,479.0
2,60,103,135,340.0
3,45,109,175,282.4
4,45,117,148,406.0
...,...,...,...,...
164,60,105,140,290.8
165,60,110,145,300.0
166,60,115,145,310.2
167,75,120,150,320.4


# Check missing data

In [16]:
data_file.shape

(169, 4)

In [17]:
data_file.dtypes

Duration      int64
Pulse         int64
Maxpulse      int64
Calories    float64
dtype: object

In [18]:
data_file.info()

<class 'pandas.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Duration  169 non-null    int64  
 1   Pulse     169 non-null    int64  
 2   Maxpulse  169 non-null    int64  
 3   Calories  164 non-null    float64
dtypes: float64(1), int64(3)
memory usage: 5.4 KB


In [19]:
data_file.describe()

,Duration,Pulse,Maxpulse,Calories
count,169.000000,169.000000,169.000000,164.000000
mean,63.846154,107.461538,134.047337,375.790244
std,42.299949,14.510259,16.450434,266.379919
min,15.000000,80.000000,100.000000,50.300000
25%,45.000000,100.000000,124.000000,250.925000
50%,60.000000,105.000000,131.000000,318.600000
75%,60.000000,111.000000,141.000000,387.600000
max,300.000000,159.000000,184.000000,1860.400000


In [20]:
data_file["Pulse"].unique()

array([110, 117, 103, 109, 102, 104,  98, 100, 106,  90,  97, 108, 130,
       105,  92, 101,  93, 107, 114, 111,  99, 123, 118, 136, 121, 115,
       153, 159, 149, 151, 129,  83,  80, 150,  95, 152, 137, 124, 116,
       112, 119, 113, 141, 122,  85, 120, 125])

In [21]:
data_file.isnull().sum()

Duration    0
Pulse       0
Maxpulse    0
Calories    5
dtype: int64

In [22]:
fig = px.scatter(
    data_file,
    x='Duration',
    y='Calories',
    color='Pulse',
    size='Maxpulse',
    hover_data=data_file.columns
)

In [23]:
fig.show()

In [24]:
corr = data_file.corr(numeric_only=True)

In [25]:
fig = px.imshow(
    corr,
    text_auto=True,
    aspect='auto'
)

fig.show()

In [26]:
out_sample_test = data_file[data_file.isnull().any(axis=1)].reset_index(drop=True).drop('Calories', axis=1)

In [27]:
out_sample_test

,Duration,Pulse,Maxpulse
0,45,90,112
1,60,103,132
2,45,107,137
3,60,105,125
4,60,97,127


In [63]:
# Define processed path
unseen_dir = Path(r"C:\Users\dooke\Projects\data-science-refresher\data\new_data")

In [64]:
# Save File
#cleaned_data.to_csv(processed_dir / "cleaned_data.csv", index=False) # index=False prevents row numbers
out_sample_test.to_csv(unseen_dir / "unseen_data.csv", 
                               index=False)

In [59]:
def transform_imputer(func):

    """

    Decorator to apply iterative imputation to numeric columns.

 

    Args:

        func: Function to be wrapped.

 

    Returns:

        Wrapped function.

    """

    def imputer_wrapper(*args, **kwargs):      

        try:    

            imputer = IterativeImputer(

                estimator=RandomForestRegressor(

                                             n_estimators=25,

                                             min_samples_split=3,

                                             max_features='log2',

                                             n_jobs=-1,

                                             random_state=10,

                                             verbose=0

                                            )

                                       )

            X = func(*args, **kwargs)

            categorical_columns=(

                                 X.select_dtypes(

                                 include=['object',

                                          'datetime64[ns]'])

                                 .columns.tolist()

                                )

            numerical_columns=(

                               X.select_dtypes(

                               include=['uint8',                                    

                                        'int64',

                                        'int32',

                                        'float64',

                                        'float32'])

                               .columns.tolist()

                               )   

            Ximputer = np.abs(

                              imputer.fit_transform(

                              X[numerical_columns])

                              )          

            Xframed = pd.DataFrame(

                                   Ximputer,

                                   index=X.index,

                                   columns=numerical_columns

                                   )

            return (

                    pd.concat(

                        [X[categorical_columns]

                        .reset_index(drop=True),

                        Xframed.reset_index(drop=True)],

                        axis=1)

                    ) 

        

        except ValueError:

            raise ValueError(f"Unsupported string/complex data found -"\

                              f" {sys.exc_info()[1]}")

        except Exception:

            raise

    return imputer_wrapper

In [60]:
@transform_imputer
def read_data_file(df) -> pd.DataFrame:
    return df

In [61]:
random_regressor_imputer = read_data_file(data_file).round(1)

In [62]:
random_regressor_imputer.isnull().sum()

Duration    0
Pulse       0
Maxpulse    0
Calories    0
dtype: int64

In [31]:
#extra_regressor_imputer[(extra_regressor_imputer['Duration']==45) & (extra_regressor_imputer['Maxpulse'].between(90, 125))]

### Data Cleansing

In [32]:
cleaned_data = data_file.dropna().reset_index(drop=True)

In [33]:
cleaned_data.head()

,Duration,Pulse,Maxpulse,Calories
0,60,110,130,409.1
1,60,117,145,479.0
2,60,103,135,340.0
3,45,109,175,282.4
4,45,117,148,406.0


### Feature Engineering

In [34]:
# Pulse × Duration - Total cardiovascular workload
cleaned_data['Pulse_Duration'] = (
    cleaned_data['Pulse'] *
    cleaned_data['Duration']
)

# Maxpulse × Duration - Peak intensity sustained over time
cleaned_data['Maxpulse_Duration'] = (
    cleaned_data['Maxpulse'] *
    cleaned_data['Duration']
)

# Exercise Intensity Ratio - i.e 0.60 = moderate exercise, 0.85 = vigorous exercise
cleaned_data['Intensity_Ratio'] = (
    cleaned_data['Pulse'] /
    cleaned_data['Maxpulse']
)

# Pulse × Maxpulse - Captures combined heart-rate load
cleaned_data['Pulse_Maxpulse'] = (
    cleaned_data['Pulse'] *
    cleaned_data['Maxpulse']
)

# Pulse rate
cleaned_data['Pulse_Squared'] = (
    cleaned_data['Pulse'] ** 2
)

# Duration rate
cleaned_data['Duration_Squared'] = (
    cleaned_data['Duration'] ** 2
)

# Heart Rate Reserve (HRR) - This captures exercise intensity better than Pulse alone.
cleaned_data['Pulse_Reserve'] = (
    cleaned_data['Maxpulse'] - cleaned_data['Pulse']
)

# Log Duration 
cleaned_data['Log_Duration'] = np.log1p(
    cleaned_data['Duration']
)

# Calories-per-Minute Proxy - Useful interactive feature
cleaned_data['Heart_Rate_Load'] = (
    cleaned_data['Duration'] *
    (cleaned_data['Pulse'] / 100)
)

In [57]:
# Define processed path
#processed_dir = Path(r"C:\Users\dooke\Projects\data-science-refresher\data\processed")

In [58]:
# Save File
#cleaned_data.to_csv(processed_dir / "cleaned_data.csv", index=False) # index=False prevents row numbers

In [35]:
corr = cleaned_data.corr(numeric_only=True)

In [36]:
fig = px.imshow(
    corr,
    text_auto=True,
    aspect='auto'
)

fig.show()

### Feature Selection

In [30]:
#selected_features = cleaned_data.drop(columns=['Maxpulse'])

## Train Test Split

In [37]:
X, Y = cleaned_data.drop(columns='Calories'), cleaned_data.loc[:, ['Calories']]

In [38]:
X.head()

,Duration,Pulse,Maxpulse,Pulse_Duration,Maxpulse_Duration,Intensity_Ratio,Pulse_Maxpulse,Pulse_Squared,Duration_Squared,Pulse_Reserve,Log_Duration,Heart_Rate_Load
0,60,110,130,6600,7800,0.846154,14300,12100,3600,20,4.110874,66.00
1,60,117,145,7020,8700,0.806897,16965,13689,3600,28,4.110874,70.20
2,60,103,135,6180,8100,0.762963,13905,10609,3600,32,4.110874,61.80
3,45,109,175,4905,7875,0.622857,19075,11881,2025,66,3.828641,49.05
4,45,117,148,5265,6660,0.790541,17316,13689,2025,31,3.828641,52.65


In [39]:
Y.shape

(164, 1)

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.20, random_state=42)

In [41]:
X_test.shape

(33, 12)

### Transform Data

In [42]:
sscaler, mmscaler, ptransformer, pca = StandardScaler(), MinMaxScaler(), PowerTransformer(), PCA()

In [43]:
# Do not fit_transform on test data because the scaler already learnt from the training data
# Golden rule - fit() = learn, transform() - apply
X_train_scaled = ptransformer.fit_transform(X_train)
X_test_scaled = ptransformer.transform(X_test)

In [65]:
import joblib

In [75]:
MODELS_DIR = ROOT_DIR / "models"

In [76]:
MODELS_DIR

WindowsPath('C:/Users/dooke/Projects/data-science-refresher/models')

In [80]:
 MODELS_DIR / "power_transformer.pkl"

WindowsPath('C:/Users/dooke/Projects/data-science-refresher/models/power_transformer.pkl')

In [79]:
joblib.dump(ptransformer, MODELS_DIR / "power_transformer.pkl")

['C:\\Users\\dooke\\Projects\\data-science-refresher\\models\\power_transformer.pkl']

In [44]:
X_train_scaled

array([[-0.4126503 , -0.29949077,  0.19894783, ...,  0.7052702 ,
        -0.41329557, -0.51673381],
       [ 0.15381082, -0.20913088,  0.13637243, ...,  0.48668239,
         0.14796888,  0.07040766],
       [-1.6741905 ,  2.41527128,  1.91220085, ..., -1.12775086,
        -1.66117936, -0.90219484],
       ...,
       [ 0.15381082, -0.68586023, -0.66945606, ..., -0.25147218,
         0.14796888, -0.02612289],
       [-2.18984138,  2.38357399,  1.91220085, ..., -1.03513037,
        -2.17404611, -1.3840676 ],
       [-1.26780652, -1.23238833, -0.32075677, ...,  0.81574875,
        -1.25865618, -1.55346378]], shape=(131, 12))

### Linear Regression

In [45]:
# Linear Regression
lr_model = LinearRegression()

# Random Forest
# Base model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Extra Trees
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# XGBoost
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

### Random Forest Hyper Porameter Tuning

In [487]:
# Hyperparameter search space
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 3, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

In [475]:
# Randomized search
rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid,
    n_iter=50,
    scoring='r2',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [476]:
# Fit hyper tuned search
rf_search.fit(X_train_scaled, y_train)

# Best model
best_rf = rf_search.best_estimator_

print("Best Parameters:")
print(rf_search.best_params_)

print("Best CV R²:")
print(rf_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


C:\Users\dooke\miniconda3\envs\dsr-env\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best Parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20, 'bootstrap': False}
Best CV R²:
0.900636041766315


In [46]:
lr_model.fit(X_train_scaled, y_train)
xgb_model.fit(X_train_scaled, y_train)
et_model.fit(X_train_scaled, y_train)
rf_model.fit(X_train_scaled, y_train)

C:\Users\dooke\miniconda3\envs\dsr-env-v2\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\dooke\miniconda3\envs\dsr-env-v2\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [47]:
# Predict
#rf_predictions_xp = best_rf.predict(X_test_scaled)

In [48]:
lr_predictions = lr_model.predict(X_test_scaled)
xgb_predictions = xgb_model.predict(X_test_scaled)
et_predictions = et_model.predict(X_test_scaled)
rf_predictions = rf_model.predict(X_test_scaled)

In [49]:
print(f"Linear Regression R² : {r2_score(y_test, lr_predictions):.4f}")

print(f"Random Forest R²     : {r2_score(y_test, rf_predictions):.4f}")

#print(f"Random Forest X R²   : {r2_score(y_test, rf_predictions_xp):.4f}")

print(f"Extra Trees R²       : {r2_score(y_test, et_predictions):.4f}")

print(f"XGBoost R²           : {r2_score(y_test, xgb_predictions):.4f}")

Linear Regression R² : 0.8625
Random Forest R²     : 0.8831
Extra Trees R²       : 0.7736
XGBoost R²           : 0.7533


In [50]:
results = {
    'Linear Regression': r2_score(y_test, lr_predictions),
    'Random Forest': r2_score(y_test, rf_predictions),
    'Extra Trees': r2_score(y_test, et_predictions),
    'XGBoost': r2_score(y_test, xgb_predictions)
}

for model, score in results.items():
    print(f'{model}: {score:.4f}')

Linear Regression: 0.8625
Random Forest: 0.8831
Extra Trees: 0.7736
XGBoost: 0.7533


In [51]:
rf_predictions

array([131.15      , 324.5       , 129.829     , 326.191     ,
       190.0295    , 198.63953333, 242.48248333, 404.454     ,
       107.747     , 341.0615    , 369.909     , 787.56966667,
       275.28444524, 128.732     , 282.2637381 , 323.163     ,
       353.138     , 575.48446667, 282.2637381 , 282.2637381 ,
       306.58057143, 240.59928333, 312.72      , 280.79371429,
       250.90225   , 111.17      , 523.13748571, 206.263     ,
       310.451     , 376.481     , 349.959     , 279.19433333,
       908.294     ])

In [52]:
result_df = y_test

In [53]:
result_df['prediction'] = rf_predictions

In [54]:
result_data = result_df.rename({'Calories': 'actual'}, axis=1).reset_index(drop=True)

In [55]:
result_data.head()

,actual,prediction
0,222.4,131.1500
1,332.7,324.5000
2,189.0,129.8290
3,305.0,326.1910
4,188.2,190.0295


In [56]:
# actual values
actual_values = result_data.actual

# predicted values
predicted_values = result_data.prediction

# create trace for actual values
actual_trace = go.Scatter(
    x=list(range(len(actual_values))),
    y=actual_values,
    mode='lines',
    name='Actual',
    line=dict(color='blue')
)

# create trace for predicted values
predicted_trace = go.Scatter(
    x=list(range(len(predicted_values))),
    y=predicted_values,
    mode='lines',
    name='Predicted',
    line=dict(color='red')
)

# create plot
fig = go.Figure(data=[actual_trace, predicted_trace])

# set title and axis labels
fig.update_layout(title='Actual vs Predicted Values',
                  xaxis_title='Index',
                  yaxis_title='Value')

# show plot
fig.show()